# Эксперименты: One-shot против Workflow

Ноутбук показывает результаты, которые сохранил `python run_experiment.py`, и помогает разобрать отдельные кейсы.
Сами прогоны в ноутбуке не запускаются (кроме последней ячейки), поэтому его можно открывать без ключа API.

In [ ]:
import json
from pathlib import Path
from IPython.display import Markdown, display

from triage.evaluate import load_cases, markdown_table, score, summarize
from triage.schemas import RunResult

cases = load_cases()
case_by_id = {c["id"]: c for c in cases}


def load_runs(name):
    data = json.loads(Path(f"results/{name}.json").read_text(encoding="utf-8"))
    return {r["case_id"]: RunResult(**r) for r in data}


oneshot = load_runs("oneshot")
workflow = load_runs("workflow")
print("Кейсов:", len(cases), "| One-shot:", len(oneshot), "| Workflow:", len(workflow))

## 1. Сравнение метрик

In [ ]:
metrics = {
    "One-shot": summarize([score(oneshot[c["id"]], c) for c in cases if c["id"] in oneshot]),
    "Workflow": summarize([score(workflow[c["id"]], c) for c in cases if c["id"] in workflow]),
}
display(Markdown(markdown_table(metrics)))

## 2. Где архитектуры ответили по-разному

Задание требует разобрать хотя бы один такой случай.

In [ ]:
different = []
for c in cases:
    a, b = oneshot.get(c["id"]), workflow.get(c["id"])
    if a is None or b is None:
        continue
    va = (a.report.incident_type, a.report.severity) if a.success else ("ошибка", "")
    vb = (b.report.incident_type, b.report.severity) if b.success else ("ошибка", "")
    if va != vb:
        different.append(c["id"])
        exp = c["expected"]
        print(f"{c['id']}: эталон {exp['incident_type']}/{exp['severity']} | one-shot {va[0]}/{va[1]} | workflow {vb[0]}/{vb[1]}")
print("Всего расхождений:", len(different))

In [ ]:
# Подробный разбор одного кейса. Можно вписать любой id из списка выше.
CASE_ID = different[0] if different else cases[0]["id"]

print("ВХОД:")
print(case_by_id[CASE_ID]["text"])
print("\nЭТАЛОН:", case_by_id[CASE_ID]["expected"])
print("\nONE-SHOT:")
print(oneshot[CASE_ID].report.model_dump_json(indent=2) if oneshot[CASE_ID].success else oneshot[CASE_ID].error)
print("\nWORKFLOW - результаты шагов:")
for step, value in workflow[CASE_ID].steps.items():
    print(f"  {step}: {value}")
print("\nWORKFLOW - итог:")
print(workflow[CASE_ID].report.model_dump_json(indent=2) if workflow[CASE_ID].success else workflow[CASE_ID].error)

## 3. Ошибки: выдуманные IOC и сбои шагов

In [ ]:
for name, runs in [("One-shot", oneshot), ("Workflow", workflow)]:
    for case_id, run in runs.items():
        if run.removed_iocs:
            print(f"{name:<9} {case_id:<28} удалены выдуманные IOC: {run.removed_iocs}")
        for problem in run.problems:
            print(f"{name:<9} {case_id:<28} {problem}")
        if not run.success:
            print(f"{name:<9} {case_id:<28} НЕУСПЕХ: {run.error}")

## 4. Дополнительное задание: искусственная ошибка одного шага

Запуск: `python run_experiment.py --fault fake_iocs` (также `ioc_step_fails` и `router_says_benign`).
Ниже выводятся сохраненные сводки.

In [ ]:
for path in sorted(Path("results").glob("fault_*.md")):
    display(Markdown(path.read_text(encoding="utf-8")))

## 5. Попробовать свой инцидент (нужен ключ в `.env`)

In [ ]:
RUN = False  # поставьте True, чтобы отправить запрос

text = """
Nov 11 02:14:09 vpn-gw01 openvpn[912]: 198.51.100.12:51820 TLS Auth Error: Auth Username/Password verification failed for peer
Nov 11 02:14:11 vpn-gw01 openvpn[912]: 198.51.100.12:51822 TLS Auth Error: Auth Username/Password verification failed for peer
Nov 11 02:14:40 vpn-gw01 openvpn[912]: 198.51.100.12:51890 user 'k.orlov' authenticated
"""
if RUN:
    from triage.llm import LLMClient
    from triage.workflow import run_workflow

    run = run_workflow(text, LLMClient())
    print(f"{run.llm_calls} вызова, {run.tokens} токенов, {run.seconds:.1f} с")
    print(run.report.model_dump_json(indent=2))